# 03 — Feature Engineering

## Goal

This notebook creates the variables I will use to analyze whether unusually high ping usage is associated with lower win rate or worse player performance.

The cleaned dataset from notebook 02 is used as the starting point.

In [32]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)

In [33]:
df = pd.read_csv(r"../data/processed/matches_clean.csv")

----------------

## 1. Create Game Length and Total Ping Features

Raw ping totals are not directly comparable across games because longer games give players more time to ping.

In [35]:
#creating total_pings, game_minutes, and pings_per_minute columns

ping_cols = [
    "all_in_pings",
    "assist_me_pings",
    "basic_pings",
    "command_pings",
    "danger_pings",
    "enemy_missing_pings",
    "enemy_vision_pings",
    "get_back_pings",
    "hold_pings",
    "need_vision_pings",
    "on_my_way_pings",
    "push_pings",
    "retreat_pings",
    "vision_cleared_pings"
]

#convert game duration from seconds to minutes
df["game_minutes"] = df["game_duration"] / 60

#total recorded pings made by each player
df["total_pings"] = df[ping_cols].sum(axis=1)

#ping rate adjusted for game length
df["pings_per_minute"] = df["total_pings"]/ df["game_minutes"]

df[["game_minutes", "total_pings", "pings_per_minute"]].describe()

,game_minutes,total_pings,pings_per_minute
count,11930.000000,11930.000000,11930.000000
mean,28.849232,31.771752,1.086024
std,6.581563,28.991405,0.925251
min,3.033333,0.000000,0.000000
25%,25.133333,12.000000,0.445054
50%,28.783333,24.000000,0.838542
75%,32.783333,42.000000,1.451272
max,49.266667,333.000000,10.493697


------------------

## 2. Create Ping-Type Rates/Shares

In [37]:
#creating per-minute columns for each ping type

ping_rate_cols = []

for col in ping_cols:
    new_col = f"{col}_per_minute"

    df[new_col] = df[col]/ df["game_minutes"]

    ping_rate_cols.append(new_col)

df[ping_rate_cols].head()

,all_in_pings_per_minute,assist_me_pings_per_minute,basic_pings_per_minute,command_pings_per_minute,danger_pings_per_minute,enemy_missing_pings_per_minute,enemy_vision_pings_per_minute,get_back_pings_per_minute,hold_pings_per_minute,need_vision_pings_per_minute,on_my_way_pings_per_minute,push_pings_per_minute,retreat_pings_per_minute,vision_cleared_pings_per_minute
0,0.037760,0.415356,0.0,0.151038,0.0,0.264317,0.113279,0.000000,0.0,0.000000,0.188798,0.0,0.075519,0.0
1,0.000000,0.377596,0.0,0.000000,0.0,0.226558,0.000000,0.000000,0.0,0.037760,0.113279,0.0,0.000000,0.0
2,0.075519,0.113279,0.0,0.302077,0.0,0.113279,0.000000,0.226558,0.0,0.075519,0.113279,0.0,0.000000,0.0
3,0.000000,0.226558,0.0,0.075519,0.0,0.075519,0.075519,0.000000,0.0,0.000000,0.453115,0.0,0.075519,0.0
4,0.000000,0.000000,0.0,0.377596,0.0,0.075519,0.377596,0.000000,0.0,0.037760,0.113279,0.0,0.037760,0.0


In [39]:
#creating ping shares (what proportion of a player's total pings come from each ping type

ping_share_cols = []

#prevents division by zero for players with no pings
total_pings_nonzero = df["total_pings"].replace(0, np.nan)

for col in ping_cols:
    new_col = f"{col}_share"

    df[new_col] = df[col]/total_pings_nonzero

    ping_share_cols.append(new_col)

df[ping_share_cols].head()

,all_in_pings_share,assist_me_pings_share,basic_pings_share,command_pings_share,danger_pings_share,enemy_missing_pings_share,enemy_vision_pings_share,get_back_pings_share,hold_pings_share,need_vision_pings_share,on_my_way_pings_share,push_pings_share,retreat_pings_share,vision_cleared_pings_share
0,0.030303,0.333333,0.0,0.121212,0.0,0.212121,0.090909,0.000000,0.0,0.000000,0.151515,0.0,0.060606,0.0
1,0.000000,0.500000,0.0,0.000000,0.0,0.300000,0.000000,0.000000,0.0,0.050000,0.150000,0.0,0.000000,0.0
2,0.074074,0.111111,0.0,0.296296,0.0,0.111111,0.000000,0.222222,0.0,0.074074,0.111111,0.0,0.000000,0.0
3,0.000000,0.230769,0.0,0.076923,0.0,0.076923,0.076923,0.000000,0.0,0.000000,0.461538,0.0,0.076923,0.0
4,0.000000,0.000000,0.0,0.370370,0.0,0.074074,0.370370,0.000000,0.0,0.037037,0.111111,0.0,0.037037,0.0


This may help identify players whose ping behavior is concentrated in one specific category.

------------------------

## 3. Define Excessive Pinging

In [40]:
df["pings_per_minute"].quantile([
    .25, .50, .75, .90, .95, .99
])

0.25    0.445054
0.50    0.838542
0.75    1.451272
0.90    2.264795
0.95    2.851629
0.99    4.361668
Name: pings_per_minute, dtype: float64

In [42]:
threshold = df["pings_per_minute"].quantile(.90)

df["excessive_pinger"] = df["pings_per_minute"] >= threshold

df['excessive_pinger'].value_counts()

excessive_pinger
False    10737
True      1193
Name: count, dtype: int64

The dataset does not directly tell me whether a player is being toxic.

Because of this, I do not label players as toxic based only on their pings.

Instead, I define an **excessive pinger** as someone whose ping rate is in the top 10% of the cleaned dataset. This gives me a consistent definition of unusually high ping activity.

I also create a percentile variable so later analysis is not limited to only comparing the top 10% with everyone else.

In [44]:
df["ping_rate_percentile"] = df["pings_per_minute"].rank(method="average",pct=True)
    
df[["pings_per_minute", "ping_rate_percentile","excessive_pinger"]].head()

,pings_per_minute,ping_rate_percentile,excessive_pinger
0,1.246067,0.685247,False
1,0.755192,0.457167,False
2,1.019509,0.591702,False
3,0.981750,0.577033,False
4,1.019509,0.591702,False


-------------------

## 4. Create Ping Activity Groups

In [45]:
q75 = df["pings_per_minute"].quantile(0.75)
q90 = df["pings_per_minute"].quantile(0.90)
q95 = df["pings_per_minute"].quantile(0.95)

In [47]:
df["ping_activity_level"] = pd.cut(
    df["pings_per_minute"],
    bins= [-np.inf, q75, q90, q95, np.inf],
    labels= ["normal","high", "very high", "extreme"])

In [48]:
df["ping_activity_level"].value_counts(sort=False)

ping_activity_level
normal       8947
high         1790
very high     596
extreme       597
Name: count, dtype: int64

----------------

## 5. Create Player Performance Features

I do not want to define player performance using only one statistic.

Instead, I create several interpretable performance variables and compare them separately later.

In [50]:
#calculating creep score (way of gold generation)

df['cs'] = df['total_minions_killed'] + df['neutral_minions_killed']

In [52]:
#performance rates adjusted for game length

df["cs_per_minute"] = df["cs"]/df["game_minutes"]

df["kills_per_minute"] = df["kills"]/df["game_minutes"]

df["deaths_per_minute"] = df["deaths"]/df["game_minutes"]

df["assists_per_minute"] = df["assists"]/df["game_minutes"]

df["gold_per_minute_calc"] = df["gold_earned"]/df["game_minutes"]

df["damage_per_minute_calc"] = df["total_damage_dealt_to_champions"]/df["game_minutes"]

df["vision_per_minute_calc"] = df["vision_score"]/df["game_minutes"]

performance_cols = ["kda", "cs_per_minute", "kills_per_minute", "deaths_per_minute", "assists_per_minute", "gold_per_minute_calc", "damage_per_minute_calc", "vision_per_minute_calc"]

----------------------------

## 6. Create Role-Adjusted Ping Percentile

In [54]:
#different roles may naturally ping at different rates(i.e. top laners may not communicate as often)

df["role_ping_percentile"] = df.groupby("team_position")["pings_per_minute"].rank(method="average",pct=True)


In [55]:
#creates another percentile that compares players in the same position

df["role_ping_percentile"] = (df.groupby("team_position")["pings_per_minute"]
    .rank(
    method="average",
    pct=True))



In [56]:
df.groupby("team_position")[["pings_per_minute", "role_ping_percentile"]].agg(
    [
        "mean",
        "median"
    ]
)

pings_per_minute           role_ping_percentile         
                          mean    median                 mean   median
team_position                                                         
BOTTOM                0.912962  0.677128              0.50021  0.50021
JUNGLE                1.492001  1.234568              0.50021  0.50021
MIDDLE                1.105487  0.879469              0.50021  0.50021
TOP                   0.781755  0.605060              0.50021  0.50021
UTILITY               1.138981  0.861750              0.50021  0.50021

`role_ping_percentile` should have mean/median around 0.5 by design since we're ranking players separately within each role.

---------------

## 7. Create Team-Level DataFrame

The player-level dataset is useful for questions about individual performance. Because all five players on the same team share the same result, I create a separate dataframe with one row per team per match.

In [57]:
team_df = (
    df.groupby(["match_id", "team_id"], as_index=False)
    .agg(
        team_win = ("win","first"),
        game_minutes = ("game_minutes", "first"),
        team_total_pings = ("total_pings", "sum"),
        team_excessive_pingers = ("excessive_pinger", "sum"),
        team_kills = ("kills", "sum"),
        team_deaths = ("deaths", "sum"),
        team_assists = ("assists", "sum"),
        team_gold = ("gold_earned", "sum"),
        team_damage = ("total_damage_dealt_to_champions", "sum")
    )
)

team_df["team_pings_per_minute"] = team_df["team_total_pings"]/team_df["game_minutes"]

team_df.head()

,match_id,team_id,team_win,game_minutes,team_total_pings,team_excessive_pingers,team_kills,team_deaths,team_assists,team_gold,team_damage,team_pings_per_minute
0,EUW1_6810099237,100,True,22.366667,184,2,38,14,47,54908,79181,8.226528
1,EUW1_6810099237,200,False,22.366667,104,1,14,38,9,39484,57954,4.649776
2,EUW1_6810123481,100,True,16.400000,142,1,17,6,16,33007,42013,8.658537
3,EUW1_6810123481,200,False,16.400000,110,1,6,17,5,27697,28267,6.707317
4,EUW1_6810162817,100,False,32.616667,334,2,38,36,57,72267,144619,10.240164


--------

## 8. Create Opponent Ping Features

In [58]:
#makes a smaller temporary dataframe containing only the columns needed to identify the opponent

opponent_df = team_df[["match_id","team_id","team_pings_per_minute"]].copy()

#renaming the column to represent the other team
opponent_df = opponent_df.rename(
    columns={
        "team_id": "opponent_team_id",
        "team_pings_per_minute": "opponent_pings_per_minute"
    }
)

#swaps the ids ((100, ping1), (200, ping2) -> (200, ping1), (100, ping2))
opponent_df["team_id"] = (
    opponent_df
    .groupby("match_id")["opponent_team_id"]
    .transform(
        lambda x: x.iloc[::-1].to_numpy()
    )
)

In [59]:
#attaches the opponent's ping rate to the original team_df

team_df = team_df.merge(
    opponent_df[["match_id", "team_id", "opponent_pings_per_minute"]],
    on=["match_id", "team_id"],
    how="left"
)

#creates team_ping_difference column
team_df["team_ping_difference"] = team_df["team_pings_per_minute"]- team_df["opponent_pings_per_minute"]

team_df.head()

,match_id,team_id,team_win,game_minutes,team_total_pings,team_excessive_pingers,team_kills,team_deaths,team_assists,team_gold,team_damage,team_pings_per_minute,opponent_pings_per_minute,team_ping_difference
0,EUW1_6810099237,100,True,22.366667,184,2,38,14,47,54908,79181,8.226528,4.649776,3.576751
1,EUW1_6810099237,200,False,22.366667,104,1,14,38,9,39484,57954,4.649776,8.226528,-3.576751
2,EUW1_6810123481,100,True,16.400000,142,1,17,6,16,33007,42013,8.658537,6.707317,1.951220
3,EUW1_6810123481,200,False,16.400000,110,1,6,17,5,27697,28267,6.707317,8.658537,-1.951220
4,EUW1_6810162817,100,False,32.616667,334,2,38,36,57,72267,144619,10.240164,5.794584,4.445580


Now each row has both my team's pings/min and my opponent's pings/min. `A positive` team_ping_difference means that team pinged more than its opponent.

This is useful to answer the question: "Does the team that pings more than its opponent tend to lose more often?" This controls for a lot of match-level conditions automatically because both teams are playing in the same game for the same duration

---------------

## 9. Sanity Check Before Saving DataFrames

In [60]:
df[[
    "game_minutes",
    "total_pings",
    "pings_per_minute",
    "ping_rate_percentile",
    "excessive_pinger",
    "cs_per_minute",
    "kills_per_minute",
    "deaths_per_minute",
    "assists_per_minute",
    "gold_per_minute_calc",
    "damage_per_minute_calc",
    "vision_per_minute_calc",
    "role_ping_percentile"
]].isna().sum()

game_minutes              0
total_pings               0
pings_per_minute          0
ping_rate_percentile      0
excessive_pinger          0
cs_per_minute             0
kills_per_minute          0
deaths_per_minute         0
assists_per_minute        0
gold_per_minute_calc      0
damage_per_minute_calc    0
vision_per_minute_calc    0
role_ping_percentile      3
dtype: int64

In [61]:
team_df[[
    "game_minutes",
    "team_total_pings",
    "team_pings_per_minute",
    "team_excessive_pingers",
    "team_kills",
    "team_deaths",
    "team_assists",
    "team_gold",
    "team_damage",
    "opponent_pings_per_minute",
    "team_ping_difference"
]].isna().sum()

game_minutes                 0
team_total_pings             0
team_pings_per_minute        0
team_excessive_pingers       0
team_kills                   0
team_deaths                  0
team_assists                 0
team_gold                    0
team_damage                  0
opponent_pings_per_minute    0
team_ping_difference         0
dtype: int64

In [62]:
#every match should have two team rows

team_df.groupby("match_id").size().value_counts()

2    1193
Name: count, dtype: int64

In [63]:
#each match should have one winning team and one losing team

team_df.groupby(
    "match_id"
)["team_win"].sum().value_counts()

team_win
1    1193
Name: count, dtype: int64

-------------

## 10. Data Error Fix

In my player level dataframe, `role_ping_percentile` contains 3 NA, likely due to missing team_position values.

In [64]:
df[df["role_ping_percentile"].isna()][["match_id", "team_position", "pings_per_minute"]]

,match_id,team_position,pings_per_minute
7233,EUW1_7439434166,NaN,0.000000
7441,EUW1_7348636740,NaN,0.775862
9603,EUW1_7340249229,NaN,0.000000


Indeed we are missing 3 values. They are from different matches so we can infer the 10th role from the other 9 players.

In [65]:
df[df["match_id"] == "EUW1_7439434166"][["team_id", "champion_name", "team_position"]]

,team_id,champion_name,team_position
7230,100,Sett,TOP
7231,100,Evelynn,JUNGLE
7232,100,Akshan,MIDDLE
7233,100,Kaisa,NaN
7234,100,Janna,UTILITY
7235,200,Sion,TOP
7236,200,Diana,JUNGLE
7237,200,Lux,MIDDLE
7238,200,Twitch,BOTTOM
7239,200,Yuumi,UTILITY


In [66]:
df[df["match_id"] == "EUW1_7348636740"][["team_id", "champion_name", "team_position"]]

,team_id,champion_name,team_position
7440,100,Teemo,TOP
7441,100,Pantheon,NaN
7442,100,Syndra,MIDDLE
7443,100,Jhin,BOTTOM
7444,100,Nami,UTILITY
7445,200,Shen,TOP
7446,200,XinZhao,JUNGLE
7447,200,Zed,MIDDLE
7448,200,Kaisa,BOTTOM
7449,200,Nautilus,UTILITY


In [67]:
df[df["match_id"] == "EUW1_7340249229"][["team_id", "champion_name", "team_position"]]

,team_id,champion_name,team_position
9600,100,Aatrox,TOP
9601,100,Rengar,JUNGLE
9602,100,Hwei,MIDDLE
9603,100,MissFortune,NaN
9604,100,Bard,UTILITY
9605,200,Jax,TOP
9606,200,Viego,JUNGLE
9607,200,Fizz,MIDDLE
9608,200,Aphelios,BOTTOM
9609,200,Nami,UTILITY


In [68]:
df.loc[
    (df["match_id"] == "EUW1_7439434166")
    & (df["team_position"].isna()),
    "team_position"
] = "BOTTOM"

df.loc[
    (df["match_id"] == "EUW1_7348636740")
    & (df["team_position"].isna()),
    "team_position"
] = "JUNGLE"

df.loc[
    (df["match_id"] == "EUW1_7340249229")
    & (df["team_position"].isna()),
    "team_position"
] = "BOTTOM"

In [69]:
df["team_position"].isna().sum()

#fixed!

np.int64(0)

-------

## 11. Save processed datasets

The player-level and team-level datasets are saved separately because they have different units of observation.

`player_features.csv` will be used for questions about individual ping behavior and individual performance.

`team_features.csv` will be used for questions about team ping behavior and win rate.

In [70]:
df.to_csv(
    "../data/processed/player_features.csv",
    index=False
)

In [71]:
team_df.to_csv(
    "../data/processed/team_features.csv",
    index=False
)

After feature engineering, several key columns/information are created to answer future analysis.
- Raw ping counts were converted to per-minute rates so games of different lengths can be compared.
- Each ping type now has both a rate and a share feature.
- Excessive pinging is defined as being in the top 10% of player ping rate.
- Player performance was represented using several different measures instead of one overall performance score.
- A role-adjusted ping percentile was created because ping behavior may naturally differ by position.
- A separate team-level dataset was created because win/loss is shared by all five players on a team.
- Opponent ping rate and team ping difference were created for direct within-match comparisons.
- High ping frequency is being used as a measure of unusually high ping activity, not as direct proof of toxicity.
- Later results should be interpreted as associations rather than proof that pinging causes worse performance.